In [ ]:
# Boot cell: makes this notebook behave exactly like `python labs/12_interrupt.py` run from the repo root.
# A Jupyter kernel has no __file__, starts in the labs folder and carries its own sys.argv; the script expects none of that.
import os, sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").is_file() and (p / "labs").is_dir())
os.chdir(_root)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
__file__ = str(_root / "labs" / "12_interrupt.py")
sys.argv = [__file__]
print("repo root:", _root)


In [ ]:
"""S12.2 Lookup is free. Refund parks. Resume with Command."""

from pathlib import Path
import sys

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from langgraph.types import Command

from dataflow.graphs.v4_hitl import build_v4_hitl


In [ ]:
graph = build_v4_hitl()
lookup_cfg = {"configurable": {"thread_id": "lookup-1"}}
look = graph.invoke({"ticket": "Status of order DF-1001?"}, lookup_cfg)
print("lookup_reply", look.get("reply"))
print("lookup_interrupts", bool(graph.get_state(lookup_cfg).tasks and False))


In [ ]:
refund_cfg = {"configurable": {"thread_id": "refund-1"}}
parked = graph.invoke({"ticket": "Please refund order DF-1001"}, refund_cfg)
print("parked_keys", sorted(parked.keys()) if parked else "interrupt")
state = graph.get_state(refund_cfg)
print("has_interrupt", bool(state.interrupts))
if state.interrupts:
    print("interrupt_action", state.interrupts[0].value.get("action"))


In [ ]:
resumed = graph.invoke(Command(resume="approve"), refund_cfg)
print("refund_decision", resumed.get("decision"))
print("refund_reply", resumed.get("reply"))
